**Eye-Tracking & Neurocognitive Intelligence | Eye-Tracking Foundations & Signal Analysis | Presented by: Reza Saadatyar (2026)**

**04 | Gaze Features and Fixation Scanpaths**

![Source-identified event features and fixation-centroid scanpath workflow](../../images/04_gaze_features_and_scanpaths.svg)

**Objective:** Convert event intervals produced in Notebook 03 into transparent per-record gaze features and an illustrative fixation-centroid path, while retaining the distinction between detector output and observed eye movement.

**Input evidence:** Notebook 02 prepared **2,778,008 samples across 40 recordings from 10 participants**. Notebook 03 recorded **7,015 I-VT fixations, 9,304 I-VT saccades and 5,409 I-DT fixations**. The present feature definitions use **I-VT events only**; I-DT remains a comparator, not a source of additional mixed event counts.

**Method scope:** Summarise validity, detected event counts and durations, peak saccade speed, approximate fixation-centroid path length and uncalibrated positive pupil area where available. This is a descriptive gaze-feature analysis, not reading-comprehension measurement, semantic word-level gaze mapping or clinical inference.

**Execution status:** Notebook 04 has not yet run on the original selected cohort. Its introductory figure combines verified prior-notebook event counts with explicitly labelled conceptual feature definitions, without fabricated Notebook 04 results.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_tracking_foundations")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# ------------------------ 5. Study Research Utilities ------------------------
from core.stage_outputs import ensure_stage_inputs
from gaze_source import read_stage
from gaze_features import record_features, feature_dictionary

# --------------------- 6. Output and Presentation Setup ----------------------
output_dir = root / "eye_tracking_foundations" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(10, 4),
)
print("Current outputs:", output_dir)


**02. Recover Prepared Samples and Detected Events**

Reuse the current `02_preprocessed_samples.csv` from Notebook 02 and `03_events.csv` from Notebook 03. The event table carries the original record and participant identities, method, event type, within-record timing and centroid coordinates.

For per-record feature calculation, the entire wide prepared sample table is unnecessary. Read only `record_id`, `participant_id`, `task`, `valid_for_motion` and `dP`, while retaining all event columns required for feature extraction. This reduces peak RAM relative to loading every gaze and velocity column again. Show the actual input scope and verify that the event record identifiers belong to the prepared sample set before computing features.

Existing files in the Colab runtime are reused; a compatible Notebook ZIP is restored only if an input is absent.


In [ ]:
# ============= 02. Recover Prepared Samples and Detected Events ==============

# ----------------------- 1. Verify Required Inputs ---------------------------
ensure_stage_inputs(output_dir, 2, ("02_preprocessed_samples.csv",))
ensure_stage_inputs(output_dir, 3, ("03_events.csv",))

# ---------------- 2. Read Only the Necessary Source Fields -------------------
source_fields = [
    "record_id", "participant_id", "task", "valid_for_motion", "dP",
]
prepared = pd.read_csv(
    output_dir / "02_preprocessed_samples.csv",
    usecols=source_fields,
    dtype={"record_id": str, "participant_id": str, "task": str},
    low_memory=False,
)
events = read_stage(
    output_dir / "03_events.csv",
    ("record_id", "participant_id", "method", "event_type",
     "duration_ms", "mean_x_dva", "mean_y_dva", "peak_speed_dva_s",
     "start_sample"),
)

# -------------------- 3. Confirm Record and Task Scope -----------------------
source_ids = set(prepared["record_id"])
if not set(events["record_id"]).issubset(source_ids):
    raise ValueError("Notebook 03 events include records absent from Notebook 02.")
if events.empty:
    raise ValueError("No detected events are available for feature extraction.")
print("Source participants:", prepared["participant_id"].nunique())
print("Source recordings:", prepared["record_id"].nunique())
print("Prepared sample rows:", f"{len(prepared):,}")
print("Detected event rows:", f"{len(events):,}")
print("Event methods:", ", ".join(sorted(events["method"].unique())))
display(events[[
    "record_id", "method", "event_type", "duration_ms", "mean_x_dva", "mean_y_dva",
]].head(8))


**03. Calculate Record-Level Features**

For each source recording, extract only **I-VT** fixation and saccade events. Report valid gaze fraction, counts of both event classes, mean and median fixation duration, and mean event-peak saccade speed. The latter is derived from provisional detector events and may be influenced by the extreme speed estimates identified in Notebook 02.

Approximate scanpath length is the sum of Euclidean distances between successive **detected fixation centroids** in degrees of visual angle. It does not include a verified word-by-word reading trajectory, nor does it necessarily account for unobserved gaze between separated valid runs. Only finite, positive pupil-area observations from motion-eligible samples enter the optional **uncalibrated** pupil summary. Missing estimates stay missing instead of being set to zero.

Save `04_features_by_record.csv` together with a compact feature dictionary documenting each feature and unit.


In [ ]:
# ==================== 03. Calculate Record-Level Features ====================

features = pd.DataFrame([
    record_features(part, events[events.record_id.eq(rid)])
    for rid, part in prepared.groupby("record_id",sort=True)
])
features.to_csv(output_dir / "04_features_by_record.csv", index=False)
feature_dictionary().to_csv(output_dir / "04_feature_dictionary.csv",index=False)
display(features)


**04. Visualise an Illustrative Fixation-Centroid Path**

Display up to the first 100 **I-VT fixation events** from a selected source recording, ordered by their original `start_sample`. Join their detected centroid coordinates in sequence and mark the first selected fixation.

The plot is a **detector-derived centroid sequence**, not an observed gaze trajectory between every fixation, a stimulus-specific area-of-interest map or a measure of reading comprehension. If the selected recording contains no qualifying fixation events, display a clear explanatory label instead of inventing a trajectory.


In [ ]:
# =========== 04. Visualise an Illustrative Fixation-Centroid Path ============

rid = str(prepared.record_id.iloc[0])
fix = events[(events.record_id==rid) & (events.method=="IVT") &
             (events.event_type=="fixation")].sort_values("start_sample").head(100)
fig, ax = plt.subplots(figsize=(7,5))
if len(fix):
    ax.plot(fix.mean_x_dva, fix.mean_y_dva, lw=0.9, marker="o", markersize=3)
    ax.scatter(fix.mean_x_dva.iloc[0], fix.mean_y_dva.iloc[0],marker="s",s=50,
               label="First selected fixation")
else:
    ax.text(0.5,0.5,"No qualifying fixations in selected record",
            ha="center",transform=ax.transAxes)
ax.set(xlabel="Gaze x (dva)",ylabel="Gaze y (dva)", title=f"Detected I-VT fixation centroids: {rid}")
ax.invert_yaxis()
ax.legend(loc="best") if len(fix) else None
fig.tight_layout()
save_figure(fig,output_dir / "04_scanpath.png")
plt.show()


**05. Review Feature Coverage**

Inspect the number of distinct participants and source recordings represented by the feature table. Missingness in a feature is informative: it may reflect no detected qualifying events, absent source pupil measurements, or an undefined statistic, not a numerical zero.

The independent unit for downstream participant-balanced uncertainty analysis is the **participant**, not every gaze sample, fixation or repeated task recording. Notebook 05 will aggregate these repeated observations before its exploratory bootstrap analysis.


In [ ]:
# ======================== 05. Review Feature Coverage ========================

print("Feature records:",len(features),"Unique participants:",features.participant_id.nunique())
print("Unavailable values by feature:")
display(features.isna().sum().rename("missing"))


**06. Export Named Current Features**

Export `04_features_by_record.csv`, `04_feature_dictionary.csv` and `04_scanpath.png`. The first two are reusable machine-readable evidence; the figure is an illustrative detected fixation-centroid path. All are newly calculated Notebook 04 outputs and should not be described as observed until this Notebook has been run.

Notebook 05 consumes the per-record features. Retain the Notebook 04 ZIP privately when switching Colab runtimes; do not commit generated CSVs by default.


In [ ]:
# ===================== 06. Export Named Current Features =====================

download_outputs_from_colab("eye_tracking_foundations/output", "04")
